In [5]:
!pip -q install ultralytics opencv-python pyyaml


In [6]:
from pathlib import Path

PROJECT_DIR = Path(r"C:/Users/HashTag/Desktop/VehicleClassificationYOLO")

DATASET_DIR = PROJECT_DIR / "data" / "vehicles"

IMAGES_TRAIN = DATASET_DIR / "images" / "train"
IMAGES_VAL   = DATASET_DIR / "images" / "val"

JSON_TRAIN = DATASET_DIR / "labels" / "train"
JSON_VAL   = DATASET_DIR / "labels" / "val"

print("DATASET_DIR:", DATASET_DIR)
print("IMAGES_TRAIN exists:", IMAGES_TRAIN.exists())
print("IMAGES_VAL exists:", IMAGES_VAL.exists())
print("JSON_TRAIN exists:", JSON_TRAIN.exists())
print("JSON_VAL exists:", JSON_VAL.exists())


DATASET_DIR: C:\Users\HashTag\Desktop\VehicleClassificationYOLO\data\vehicles
IMAGES_TRAIN exists: True
IMAGES_VAL exists: True
JSON_TRAIN exists: True
JSON_VAL exists: True


In [8]:
CLASS_NAMES = ["car", "bus", "truck", "motorcycle"]

NAME_ALIASES = {
    "motorbike": "motorcycle",
    "bike": "motorcycle",
    "van": "car",
}

name_to_id = {n: i for i, n in enumerate(CLASS_NAMES)}
print("name_to_id:", name_to_id)


name_to_id: {'car': 0, 'bus': 1, 'truck': 2, 'motorcycle': 3}


In [9]:
import json
from collections import defaultdict
import cv2

IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

def find_image_by_stem(img_dir: Path, stem: str):
    for ext in IMG_EXTS:
        p = img_dir / f"{stem}{ext}"
        if p.exists():
            return p
    for p in img_dir.iterdir():
        if p.is_file() and p.stem == stem and p.suffix.lower() in IMG_EXTS:
            return p
    return None

def normalize_name(n: str) -> str:
    n2 = n.strip().lower()
    return NAME_ALIASES.get(n2, n2)

def xyxy_to_yolo(xmin, ymin, xmax, ymax, w, h):
    xmin = max(0, min(xmin, w))
    xmax = max(0, min(xmax, w))
    ymin = max(0, min(ymin, h))
    ymax = max(0, min(ymax, h))
    bw = max(0.0, xmax - xmin)
    bh = max(0.0, ymax - ymin)
    cx = xmin + bw / 2.0
    cy = ymin + bh / 2.0
    return cx / w, cy / h, bw / w, bh / h

def write_yolo_txt(out_path: Path, rows):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for cls_id, x, y, bw, bh in rows:
            f.write(f"{cls_id} {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}\n")

def detect_json_type(obj: dict):
    if "images" in obj and "annotations" in obj and "categories" in obj:
        return "coco"
    if "shapes" in obj and "imageWidth" in obj and "imageHeight" in obj:
        return "labelme"
    if "bboxes" in obj or "bbox" in obj or "objects" in obj:
        return "generic"
    return "unknown"

def convert_coco(coco: dict, images_dir: Path, out_dir: Path):
    cat_id_to_name = {c["id"]: normalize_name(c.get("name","")) for c in coco["categories"]}
    img_id_to_file = {im["id"]: im.get("file_name") for im in coco["images"]}
    img_id_to_size = {im["id"]: (im.get("width"), im.get("height")) for im in coco["images"]}

    anns_by_img = defaultdict(list)
    for ann in coco["annotations"]:
        anns_by_img[ann["image_id"]].append(ann)

    written = 0
    for img_id, file_name in img_id_to_file.items():
        if not file_name:
            continue
        w, h = img_id_to_size.get(img_id, (None, None))
        if not w or not h:
            img_path = find_image_by_stem(images_dir, Path(file_name).stem)
            if img_path is None:
                continue
            im = cv2.imread(str(img_path))
            h, w = im.shape[:2]

        stem = Path(file_name).stem
        rows = []
        for ann in anns_by_img.get(img_id, []):
            name = cat_id_to_name.get(ann.get("category_id"), "")
            if name not in name_to_id:
                continue
            bbox = ann.get("bbox")  # COCO xywh
            if not bbox or len(bbox) != 4:
                continue
            x, y, bw, bh = bbox
            xmin, ymin, xmax, ymax = x, y, x+bw, y+bh
            xc, yc, ww, hh = xyxy_to_yolo(xmin, ymin, xmax, ymax, w, h)
            rows.append((name_to_id[name], xc, yc, ww, hh))

        if rows:
            write_yolo_txt(out_dir / f"{stem}.txt", rows)
            written += 1
    print("COCO conversion wrote:", written, "txt files")

def convert_labelme(j: dict, json_path: Path, out_dir: Path):
    w, h = j["imageWidth"], j["imageHeight"]
    rows = []
    for sh in j["shapes"]:
        label = normalize_name(sh.get("label",""))
        if label not in name_to_id:
            continue
        pts = sh.get("points", [])
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        xmin, xmax = min(xs), max(xs)
        ymin, ymax = min(ys), max(ys)
        xc, yc, ww, hh = xyxy_to_yolo(xmin, ymin, xmax, ymax, w, h)
        rows.append((name_to_id[label], xc, yc, ww, hh))

    if rows:
        write_yolo_txt(out_dir / f"{json_path.stem}.txt", rows)
        return 1
    return 0

def convert_generic(j: dict, json_path: Path, images_dir: Path, out_dir: Path):
    img_path = find_image_by_stem(images_dir, json_path.stem)
    if img_path is None:
        return 0
    im = cv2.imread(str(img_path))
    h, w = im.shape[:2]

    objs = []
    if isinstance(j.get("bboxes"), list): objs = j["bboxes"]
    elif isinstance(j.get("objects"), list): objs = j["objects"]
    elif isinstance(j.get("bbox"), list): objs = [{"label": j.get("label",""), "bbox": j["bbox"]}]

    rows = []
    for o in objs:
        label = normalize_name(o.get("label") or o.get("class") or "")
        if label not in name_to_id:
            continue
        bbox = o.get("bbox")
        if not bbox or len(bbox) != 4:
            continue

        # guess xyxy first, fallback xywh
        x1,y1,x2,y2 = bbox
        if x2 < x1 or y2 < y1:  # looks like xywh
            x,y,bw,bh = bbox
            xmin,ymin,xmax,ymax = x,y,x+bw,y+bh
        else:
            xmin,ymin,xmax,ymax = x1,y1,x2,y2

        xc,yc,ww,hh = xyxy_to_yolo(xmin,ymin,xmax,ymax,w,h)
        rows.append((name_to_id[label], xc, yc, ww, hh))

    if rows:
        write_yolo_txt(out_dir / f"{json_path.stem}.txt", rows)
        return 1
    return 0

def convert_folder(json_dir: Path, images_dir: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    json_files = sorted(list(json_dir.glob("*.json")))
    if not json_files:
        print("No json files in:", json_dir)
        return

    # COCO single-file case
    if len(json_files) == 1:
        obj = json.loads(json_files[0].read_text(encoding="utf-8"))
        if detect_json_type(obj) == "coco":
            convert_coco(obj, images_dir, out_dir)
            return

    # per-image jsons
    written = 0
    unknown = []
    for jp in json_files:
        obj = json.loads(jp.read_text(encoding="utf-8"))
        t = detect_json_type(obj)
        if t == "labelme":
            written += convert_labelme(obj, jp, out_dir)
        elif t == "generic":
            written += convert_generic(obj, jp, images_dir, out_dir)
        else:
            if len(unknown) < 3:
                unknown.append((jp.name, list(obj.keys())[:25]))
    print("Per-image conversion wrote:", written, "txt files")
    if unknown:
        print("\n Unknown JSON schema samples (file -> keys):")
        for fn, keys in unknown:
            print(" -", fn, "->", keys)
        print("If you see this, send ONE json file content here and I will adapt the converter.")
